# MobileNetV2 Transfer Learning (Flower Photos)
Run the cells in order. Designed for Google Colab.

## 1. Install packages

In [ ]:
!pip install -q split-folders

## 2. Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import splitfolders

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

## 3. Download the dataset

In [ ]:
url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir = tf.keras.utils.get_file("flower_photos", origin=url, untar=True)

# Find the folder that actually contains the class folders
if not os.path.isdir(os.path.join(data_dir, "daisy")):
    data_dir = os.path.join(data_dir, "flower_photos")

print("Dataset folder:", data_dir)
print(os.listdir(data_dir))

## 4. Split into train / val / test (70% / 15% / 15%)

In [ ]:
OUTPUT_DIR = "/content/dataset"

if not os.path.exists(OUTPUT_DIR):
    splitfolders.ratio(data_dir, output=OUTPUT_DIR, seed=42, ratio=(0.7, 0.15, 0.15))

print(os.listdir(OUTPUT_DIR))  # ['train', 'val', 'test']

## 5. Parameters

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

BASE_PATH = "/content/dataset"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
VALIDATION_PATH = os.path.join(BASE_PATH, "val")
TEST_PATH = os.path.join(BASE_PATH, "test")

## 6. Data preprocessing and loading

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)
validation_datagen = ImageDataGenerator(rescale=1.0 / 255)
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_data = train_datagen.flow_from_directory(
    TRAIN_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=True)

validation_data = validation_datagen.flow_from_directory(
    VALIDATION_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False)

test_data = test_datagen.flow_from_directory(
    TEST_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False)

num_classes = train_data.num_classes
print("\nNumber of classes:", num_classes)
print("Class names:", train_data.class_indices)

## 7. Build the model (MobileNetV2 + custom classifier)

In [ ]:
base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(128, 128, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(num_classes, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

## 8. Train the custom classifier

In [ ]:
history1 = model.fit(train_data, validation_data=validation_data, epochs=5)

## 9. Evaluate before fine-tuning

In [ ]:
test_loss1, test_accuracy1 = model.evaluate(test_data)
print("\nTest Loss Before Fine-Tuning:", test_loss1)
print("Test Accuracy Before Fine-Tuning:", test_accuracy1)

## 10. Fine-tuning (unfreeze last 20 layers, keep BatchNorm frozen)

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

for layer in base_model.layers[-20:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history2 = model.fit(train_data, validation_data=validation_data, epochs=5)

## 11. Final evaluation

In [ ]:
test_loss2, test_accuracy2 = model.evaluate(test_data)
print("\nFinal Test Loss:", test_loss2)
print("Final Test Accuracy:", test_accuracy2)

## 12. Combine training history

In [ ]:
accuracy = history1.history["accuracy"] + history2.history["accuracy"]
val_accuracy = history1.history["val_accuracy"] + history2.history["val_accuracy"]
loss = history1.history["loss"] + history2.history["loss"]
val_loss = history1.history["val_loss"] + history2.history["val_loss"]

## 13. Plot accuracy and loss

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(accuracy, label="Training Accuracy")
plt.plot(val_accuracy, label="Validation Accuracy")
plt.axvline(x=4.5, linestyle="--", color="gray", label="Fine-Tuning Starts")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("MobileNetV2 Transfer Learning Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(loss, label="Training Loss")
plt.plot(val_loss, label="Validation Loss")
plt.axvline(x=4.5, linestyle="--", color="gray", label="Fine-Tuning Starts")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("MobileNetV2 Transfer Learning Loss")
plt.legend()
plt.show()

## 14. (Optional) Confusion matrix and classification report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

test_data.reset()
pred = np.argmax(model.predict(test_data), axis=1)
labels = list(test_data.class_indices.keys())

print(classification_report(test_data.classes, pred, target_names=labels))
print(confusion_matrix(test_data.classes, pred))